<a href="https://colab.research.google.com/github/ProfessorPatrickSlatraigh/CIS3120-BMWB/blob/main/CIS3120_Module15_PressRelease2Plot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 15 — Press Release to Plot

**Course:** CIS 3120 — Programming for Analytics
**Pipeline:** SEC EDGAR full-text search → Anthropic API extraction → folium map
**Shortlink to this notebook:** `bit.ly/cis3120_module15`

This notebook retrieves 8-K filings from the past 30 days that contain location-event keywords, uses Claude Haiku 4.5 to classify and extract structured location data from each filing, and plots the results on an interactive map.

The pipeline is deliberately broken into five inspectable stages. After each stage, the next cell prints a summary of what was retrieved or produced. This lets you stop and examine intermediate output, which is the most reliable way to understand what each step is contributing to the final result.

## How to work through this notebook

You will complete three exercises as you walk through the pipeline:

- **Exercise S1 (Stage 1):** Implement the EDGAR search function.
- **Exercise S3 (Stage 3):** Write the system prompt that elicits structured JSON from Claude.
- **Exercise S5 (Stage 5):** Build the folium marker loop with color encoding and popup HTML.

Stages 2 and 4 are provided complete so that you can focus on the parts of the pipeline that exercise the new concepts in this module.

## Setup

The pipeline depends on four external libraries:

- `requests` — issues HTTP calls to SEC EDGAR and to OpenStreetMap
- `anthropic` — official Python SDK for the Claude API
- `folium` — Python wrapper that produces interactive Leaflet maps
- `beautifulsoup4` — parses HTML content from EDGAR exhibits

Of these, only `anthropic` and `folium` are guaranteed to require installation in a fresh Colab runtime.

In [ ]:
!pip install -q anthropic folium beautifulsoup4 requests

### Storing the API key in Colab Secrets

Anthropic API keys are sensitive credentials. Pasting one directly into a notebook cell creates real risks: the key becomes part of the notebook file, gets committed to GitHub, appears in screenshots shared during help sessions, and remains visible to anyone who opens the notebook later.

The standard solution in Colab is the Secrets manager (key icon in the left sidebar). Stored secrets are accessible only to your account and only when the notebook is run interactively. The pattern below reads the secret named `ANTHROPIC_API_KEY` and places it into the environment variable that the Anthropic SDK looks for automatically.

In [ ]:
import os
from google.colab import userdata

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

### Identifying header for outbound requests

Both SEC EDGAR and the OpenStreetMap geocoding service require a descriptive `User-Agent` header that identifies who is making the request. Failing to provide one results in a `403 Forbidden` response.

> **Required action.** Replace the placeholder string below with your actual name and Baruch email address before running the cell. Submissions that retain the placeholder will not run successfully against EDGAR.

In [ ]:
# REQUIRED: replace with your name and Baruch email address
USER_AGENT = "CIS3120 Student your.name@baruchmail.cuny.edu"

REQUEST_HEADERS = {"User-Agent": USER_AGENT}

### A note on API etiquette

Public, no-cost APIs such as SEC EDGAR and OpenStreetMap Nominatim are operated as a service to researchers, journalists, and developers. They have no commercial revenue from individual users. To remain available, they impose two basic requirements:

1. **Identification.** Every request must carry a `User-Agent` header with a real name and contact email. This allows the service to contact a misbehaving client before resorting to an IP block.
2. **Rate limiting.** Each service publishes a maximum request rate. SEC EDGAR allows ten requests per second; OpenStreetMap Nominatim allows one request per second. Code that respects these limits with explicit `time.sleep()` calls is responsible practice and protects the resource for other users.

Violating these expectations typically results in a temporary IP block. For commercial workloads, paid services with higher rate limits and stronger guarantees are available, but for academic and prototype work the public services are sufficient.

## Stage 1 — Retrieve candidate 8-K filings from EDGAR

The first stage casts a wide net: it asks SEC EDGAR for every 8-K filing in the past 30 days that contains any one of a list of location-event keywords. Many of these candidates will turn out to be false positives — that is expected and is exactly why Stage 3 exists. The goal here is recall, not precision.

### Background: what EDGAR is and why this data exists

EDGAR (Electronic Data Gathering, Analysis, and Retrieval) is the SEC's public filing system. United States federal securities law requires publicly traded companies to disclose material events to the investing public on a continuous basis. Form 8-K is the specific filing used to report unscheduled material events between quarterly reports — including (relevant for this pipeline) the opening, closing, relocation, or expansion of significant facilities.

EDGAR has been online since 1994 and contains millions of filings. The full-text search endpoint at `efts.sec.gov/LATEST/search-index` is free, requires no API key, and indexes filings back to 2001 including their attached exhibits. Press releases announcing material events are typically attached to 8-K filings as Exhibit 99.1, which is the file the next stage will retrieve.

In [ ]:
import requests
import time
from datetime import date, timedelta

# Each phrase is queried independently, then results are merged.
# This avoids a parser limitation in the SEC's full-text search engine
# that prevents combining many quoted phrases with parenthetical OR groups.
SEARCH_PHRASES = [
    '"new store"',
    '"new facility"',
    '"new location"',
    '"distribution center"',
    '"fulfillment center"',
    '"grand opening"',
    '"store closure"',
    '"store closing"',
    '"facility closure"',
    '"opens new"',
]

# 30-day window ending today
WINDOW_DAYS = 30
WINDOW_END = date.today()
WINDOW_START = WINDOW_END - timedelta(days=WINDOW_DAYS)

EDGAR_SEARCH_URL = "https://efts.sec.gov/LATEST/search-index"

print(f"Search window: {WINDOW_START} to {WINDOW_END} ({WINDOW_DAYS} days)")
print(f"Phrases to query: {len(SEARCH_PHRASES)}")

### Why we run one query per phrase instead of one combined query

A natural first attempt is to combine all phrases into a single query using boolean OR — for example, `"new store" OR "new facility" OR "distribution center"`. The SEC's documentation suggests this should work, but in practice the query parser is more restricted than the documentation describes. Combining many quoted phrases with parentheses produces zero hits, even for queries where each phrase individually returns hundreds of matches.

The robust workaround is to issue one search per phrase and merge the results. Each filing is uniquely identified by its accession number (the `_id` field on each search hit), so deduplication is straightforward: a Python dictionary keyed on accession number naturally collapses any filing that matches multiple phrases.

This pattern — discover a documented feature is broken in practice, then build a more resilient workaround — is a common situation in real analytical work. Documentation describes intent; observed behavior is the truth.

### Exercise S1 — Build the EDGAR query function

Implement `search_edgar_one_phrase(phrase, start_date, end_date, forms, max_pages)` so that it issues one or more paginated GET requests to the EDGAR full-text search endpoint and returns a tuple of `(hits_list, total_available_in_window)`.

**Required behavior:**

1. Build a `params` dictionary with the keys `q`, `dateRange`, `startdt`, `enddt`, `forms`, and `from`. The first five take the obvious values; `from` is the pagination offset (`page * 100`).
2. Issue the request with `requests.get(EDGAR_SEARCH_URL, params=params, headers=REQUEST_HEADERS, timeout=30)`. Call `response.raise_for_status()` to fail loudly on HTTP errors.
3. Parse the JSON response. Hits are at `data["hits"]["hits"]`; total available is at `data["hits"]["total"]["value"]`.
4. Accumulate hits across pages. Stop early when `(page + 1) * 100 >= total`.
5. Pause `0.15` seconds between paginated requests to respect SEC's ten-requests-per-second limit.
6. Return `(all_hits, total)`.

The function signature, docstring, and a `raise NotImplementedError` placeholder are provided. Replace the placeholder with your implementation.

In [ ]:
def search_edgar_one_phrase(phrase, start_date, end_date, forms="8-K", max_pages=2):
    """Run a single-phrase EDGAR full-text search.

    Returns a tuple of (hits_list, total_available_in_window). The endpoint
    paginates with up to 100 hits per page; we cap at max_pages pages.
    """
    all_hits = []
    total = 0

    # TODO 1: Loop over pages from 0 up to max_pages.
    # TODO 2: For each page, build the params dict with q, dateRange, startdt,
    #         enddt, forms, and from (= page * 100).
    # TODO 3: Issue the GET request with REQUEST_HEADERS and timeout=30, and
    #         call response.raise_for_status().
    # TODO 4: Extract hits from data["hits"]["hits"] and append to all_hits.
    #         Update total from data["hits"]["total"]["value"].
    # TODO 5: Break out of the loop early if (page + 1) * 100 >= total.
    # TODO 6: time.sleep(0.15) between paginated requests.

    raise NotImplementedError("Complete Exercise S1 to enable the next cell.")

    return all_hits, total

# ══ SOLUTION ══

In [ ]:
# Run one query per phrase, deduplicate results by accession ID
all_hits_by_id = {}
print(f"{'Phrase':<25} {'Total in window':>18}")
print("-" * 45)
for phrase in SEARCH_PHRASES:
    hits, total = search_edgar_one_phrase(phrase, WINDOW_START, WINDOW_END)
    print(f"{phrase:<25} {total:>18}")
    for hit in hits:
        all_hits_by_id[hit["_id"]] = hit
    time.sleep(0.15)

candidates = list(all_hits_by_id.values())
total_found = len(candidates)
print(f"\nUnique candidates after deduplication: {total_found}")

### Inspect the candidate set

Before sending anything to the LLM, look at the raw retrieval. Each EDGAR hit contains the company name, filing date, accession number, and the exhibit filename containing the press release text. A quick visual scan often reveals patterns — for example, a single company filing many 8-Ks in a short window, or filings from companies whose names suggest they probably do not operate physical facilities.

In [ ]:
# Show the first 10 candidates so we can visually inspect what came back
print(f"{'Date':<12} {'Company':<45} {'Exhibit'}")
print("-" * 100)
for hit in candidates[:10]:
    src = hit["_source"]
    company = (src.get("display_names") or ["(unknown)"])[0]
    if len(company) > 43:
        company = company[:42] + "…"
    file_date = src.get("file_date", "")
    exhibit = hit["_id"].split(":")[-1] if ":" in hit["_id"] else hit["_id"]
    print(f"{file_date:<12} {company:<45} {exhibit}")

## Stage 2 — Fetch the press release text from each filing

The search results from Stage 1 contain metadata about each filing but not the press release text itself. This stage downloads each exhibit file and converts it to clean plain text suitable for the LLM in Stage 3.

Stage 2 is provided complete. You do not need to modify any code in this stage.

### Background: what an "exhibit" is and how we locate the file

An 8-K filing is structured: a short cover document declares which "items" the filing reports (Item 8.01 for "Other Events," Item 2.05 for "Costs Associated with Exit Activities," and so on), and one or more exhibits provide the supporting material. For announcements of facility openings or closings, the substantive content almost always lives in **Exhibit 99.1**, which is the press release the company's communications team issued.

EDGAR stores every filing under a stable URL pattern:

```
https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}
```

The CIK (Central Index Key) is the SEC's unique identifier for each company, and the accession number is the unique identifier for each filing. Both are present in the search hit returned by Stage 1, which is what `build_exhibit_url()` below assembles into a complete URL.

In [ ]:
from bs4 import BeautifulSoup


def build_exhibit_url(hit):
    """Construct the canonical EDGAR archive URL for the exhibit."""
    accession_full, filename = hit["_id"].split(":")
    accession_no_dashes = accession_full.replace("-", "")
    cik = hit["_source"]["ciks"][0].lstrip("0")
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession_no_dashes}/{filename}"


def fetch_exhibit_text(hit, max_chars=8000):
    """Download the exhibit, strip HTML to plain text, and truncate.

    Returns (clean_text, exhibit_url).
    """
    url = build_exhibit_url(hit)
    response = requests.get(url, headers=REQUEST_HEADERS, timeout=30)
    response.raise_for_status()

    # Convert HTML markup to plain text
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)

    # Truncate long releases to control LLM input cost in Stage 3
    if len(text) > max_chars:
        text = text[:max_chars] + " […truncated…]"

    return text, url


# Smoke test: fetch the first candidate and inspect the first 500 characters
sample_text, sample_url = fetch_exhibit_text(candidates[0])
print(f"URL: {sample_url}")
print(f"Length: {len(sample_text)} characters")
print(f"\nFirst 500 characters:\n{sample_text[:500]}")

### Why HTML stripping and truncation are needed

EDGAR serves exhibits in HTML so they render correctly in a browser. For an LLM that processes plain text, HTML markup is noise: it inflates the token count without adding meaning, and complex layouts can confuse the model about which text is the substantive content versus boilerplate (footers, page numbers, exhibit indexes).

The `BeautifulSoup` library walks the parsed HTML tree and extracts only the visible text content, discarding tags and inline styling. The result is the press release as a continuous block of plain text.

The 8,000-character truncation is a deliberate cost-control measure. Most press releases are between 1,500 and 4,000 characters and pass through unchanged; the truncation only affects unusually long filings such as multi-event announcements or attached financial tables. Eight thousand characters maps to roughly 2,000 tokens — enough context for the LLM to determine event type and location, while keeping per-filing input cost predictable.

### Fetch all candidate exhibits

The loop below fetches every candidate. With approximately 100 candidates and a 0.15-second pause per request (well under the SEC's published limit of ten requests per second), the loop completes in roughly thirty seconds. Failures are logged but do not halt the pipeline.

In [ ]:
fetched_filings = []
fetch_failures = 0

for i, hit in enumerate(candidates):
    try:
        text, url = fetch_exhibit_text(hit)
        src = hit["_source"]
        fetched_filings.append({
            "company": (src.get("display_names") or ["(unknown)"])[0],
            "ticker": (src.get("tickers") or [None])[0],
            "file_date": src.get("file_date"),
            "accession": hit["_id"].split(":")[0],
            "url": url,
            "text": text,
        })
    except Exception as exc:
        fetch_failures += 1
        print(f"  [warn] Failed to fetch {hit['_id']}: {exc}")

    time.sleep(0.15)  # respect SEC's 10 req/sec rate limit

    if (i + 1) % 25 == 0:
        print(f"Fetched {i + 1} of {len(candidates)}…")

print(f"\nFetched {len(fetched_filings)} filings successfully.")
print(f"Failures: {fetch_failures}")

## Stage 3 — Classify and extract with the Anthropic API

For each fetched filing, the pipeline asks Claude two questions in a single call: (1) does this filing genuinely describe a location-related event, and (2) if so, what are the structured details (event type, city, state, brief summary). Filings that fail the first check are discarded; the rest are passed to Stage 4 for geocoding.

### Background: asking the model for structured output

Module 14 demonstrated how to send free-form prompts to Claude and receive free-form text responses. For analytical pipelines this is rarely useful — downstream code needs values it can address by name, not paragraphs of prose to parse.

The technique here is **structured output via prompt engineering**: the system prompt explicitly instructs the model to return only a JSON object with a specified schema. The downstream code parses that JSON with `json.loads()` and accesses fields directly (`extraction["event_type"]`, `extraction["city"]`, and so on).

Two practical considerations when designing this kind of prompt:

1. **The schema must be unambiguous.** Specify the exact field names, the type of each field (string, boolean, null), and the allowed values for any enumerated field. Vague specifications produce inconsistent output.
2. **Defensive parsing matters.** Even with explicit instructions, models occasionally wrap JSON in markdown fences, add a one-line preamble, or hallucinate an extra field. The `re.sub` line in the function below strips markdown fences if they appear; a `try/except` around `json.loads()` catches the rare malformed response without halting the loop.

### Why Haiku 4.5 is the right model for this task

Module 14 used Claude Opus 4.7 for general-purpose generation. Opus is the most capable model in the Claude family, but capability is not the only consideration — for high-volume, well-bounded tasks, a smaller model is the better engineering choice.

This pipeline performs a binary classification (is this a location event, yes or no) followed by structured extraction of a handful of fields from short text. That workload is precisely what **Claude Haiku 4.5** is designed for. At one dollar per million input tokens (compared to five dollars for Opus), it reduces the per-run cost by approximately eighty percent while producing equivalent results on this kind of well-specified task. Using Opus here would be analogous to renting a freight truck to deliver a single envelope — the smaller vehicle is the more responsible choice when the workload does not justify the larger one.

The general principle: match the model to the task. Reserve the most capable models for the parts of a pipeline that genuinely require deep reasoning, and use smaller, cheaper models for routine extraction and classification.

### Exercise S3 — Write the extraction system prompt

This is the highest-leverage exercise in the notebook. The system prompt is the single artifact that determines whether the LLM stage succeeds or fails. A vague prompt produces inconsistent JSON; a strict prompt produces clean, parseable records that the rest of the pipeline can rely on.

Your task is to write `EXTRACTION_SYSTEM_PROMPT` as a Python string. The prompt must instruct Claude to:

1. Adopt the role of an analyst reviewing 8-K filing exhibits to identify location-related corporate events (openings, closings, relocations, expansions of physical facilities such as stores, warehouses, distribution centers, offices, plants).
2. Return **only** a JSON object with these exact fields:
   - `is_location_event` — boolean. True only when the filing genuinely announces opening, closing, relocation, or expansion of a specific physical facility at a named location. False for earnings, executive changes, financing, share repurchases, or generic corporate updates.
   - `event_type` — one of `"opening"`, `"closing"`, `"relocation"`, `"expansion"`, `"other"`, or null.
   - `city` — string with the city name, or null.
   - `state` — two-letter US state code (e.g. `"NY"`, `"CA"`), or null if not US-based or not specified.
   - `summary` — one sentence under 25 words describing the event in plain language, or null.
3. Be strict: locations mentioned only in passing (such as the headquarters address in boilerplate) should yield `is_location_event: false`.
4. Return only the JSON object — no preamble, no markdown fences, no explanation.

The function shell `extract_with_claude()` and the user-message format are provided. Write the system prompt only.

In [ ]:
import anthropic
import json
import re

client = anthropic.Anthropic()
MODEL_ID = "claude-haiku-4-5-20251001"

# TODO: Replace the placeholder below with your extraction system prompt.
# Refer to the four numbered requirements in the cell above.
EXTRACTION_SYSTEM_PROMPT = """TODO: write the system prompt here."""


def extract_with_claude(filing):
    """Send filing text to Claude and parse the JSON response."""
    user_message = f"Filing from {filing['company']} dated {filing['file_date']}:\n\n{filing['text']}"

    response = client.messages.create(
        model=MODEL_ID,
        max_tokens=400,
        system=EXTRACTION_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_message}],
    )

    raw = response.content[0].text.strip()

    # Defensive parsing: strip markdown fences if the model added them
    raw = re.sub(r"^```(?:json)?|```$", "", raw, flags=re.MULTILINE).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"is_location_event": False, "_parse_error": raw[:200]}

# ══ SOLUTION ══

In [ ]:
# Smoke test on the first filing
sample_extraction = extract_with_claude(fetched_filings[0])
print(json.dumps(sample_extraction, indent=2))

### The recall vs precision tradeoff that this pipeline manages

This is a useful moment to introduce a concept that recurs throughout analytics. Information retrieval systems are typically evaluated on two complementary measures:

- **Recall** is the fraction of relevant items the system actually retrieves. If twenty filings in the past 30 days truly describe new store openings and the system finds eighteen of them, recall is 90 percent.
- **Precision** is the fraction of retrieved items that are actually relevant. If the system returns 100 candidates and only 18 are real openings, precision is 18 percent.

These two measures usually trade off against each other. Tightening the keyword list raises precision but loses some real events (lower recall). Loosening it raises recall but admits more false positives (lower precision).

The pipeline you are running uses a deliberate division of labor:

1. **Stage 1 optimizes for recall.** Broad keyword search catches almost every relevant filing, accepting that many irrelevant ones come along.
2. **Stage 3 supplies the precision.** The LLM reads each candidate and discards the ones that are not genuine location events.

When you run the next cell, watch the ratio in the classification summary. A typical result is approximately 12 to 18 percent of candidates classified as genuine events. That percentage is not a problem with the pipeline — it is the precision the LLM contributes, and the demonstration that the LLM stage is doing real work.

### Run extraction on every filing

The loop below processes one filing per API call. With approximately 100 candidates at Haiku 4.5 pricing, the full run costs less than fifty cents. The `max_filings` cap remains as a safety bound; raise or lower it to control how many filings are processed.

In [ ]:
max_filings = 250  # Haiku makes the full candidate set affordable

events = []
classification_counts = {"location_event": 0, "not_location_event": 0, "error": 0}

for i, filing in enumerate(fetched_filings[:max_filings]):
    try:
        extraction = extract_with_claude(filing)
        if extraction.get("is_location_event"):
            classification_counts["location_event"] += 1
            events.append({**filing, **extraction})
        else:
            classification_counts["not_location_event"] += 1
    except Exception as exc:
        classification_counts["error"] += 1
        print(f"  [warn] Extraction failed for {filing['company']}: {exc}")

    if (i + 1) % 25 == 0:
        print(f"Processed {i + 1} of {min(max_filings, len(fetched_filings))}…")

print(f"\nClassification summary:")
for label, count in classification_counts.items():
    print(f"  {label}: {count}")
print(f"\nLocation events to plot: {len(events)}")

### Inspect the extracted events

Before geocoding, review a sample of what the LLM identified. This is the moment to spot-check that the classifications make sense — a quick scan can reveal whether the model is consistently catching real events and consistently rejecting false positives, or whether the prompt needs adjustment.

In [ ]:
for event in events[:10]:
    print(f"{event['file_date']} | {event['company']}")
    print(f"  Type:    {event['event_type']}")
    print(f"  Where:   {event.get('city')}, {event.get('state')}")
    print(f"  Summary: {event['summary']}")
    print()

## Stage 4 — Geocode the locations

The structured records from Stage 3 contain city and state names but not coordinates. Folium plots markers from latitude and longitude, so each location must first be converted to a coordinate pair. This step is called **geocoding**.

Stage 4 is provided complete. You do not need to modify any code in this stage.

### Background: what geocoding is and which service we use

Geocoding is the process of converting a human-readable place description ("Memphis, TN") into geographic coordinates (latitude 35.149, longitude -90.049). The reverse operation — coordinates to address — is called reverse geocoding.

Several geocoding services are available. Google Maps and Mapbox offer high-quality geocoding with permissive rate limits, but both require API keys and become expensive at volume. The free alternative, used here, is **OpenStreetMap Nominatim**, which is operated by the OpenStreetMap Foundation as a public service. Nominatim is free and requires no API key, but it enforces a strict limit of one request per second and requires a descriptive `User-Agent` header.

Two failure modes are common with city-level geocoding and worth being aware of:

1. **Ambiguous city names.** Many cities share names with other places — "Springfield" alone is unresolvable because it exists in many states. Including the state in the query, as the function below does, almost always resolves this.
2. **Non-US locations.** The pipeline appends `, USA` to every query for consistency. Filings about international facilities will fail to geocode and are dropped at this stage. For an internationalized version of the lesson, the city-state-country logic would need to be more flexible.

In [ ]:
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"


def geocode_location(city, state):
    """Convert a city/state pair to (latitude, longitude). Returns None on failure."""
    if not city:
        return None

    query = f"{city}, {state}, USA" if state else f"{city}, USA"
    params = {"q": query, "format": "json", "limit": 1}

    try:
        response = requests.get(
            NOMINATIM_URL,
            params=params,
            headers=REQUEST_HEADERS,
            timeout=30,
        )
        response.raise_for_status()
        results = response.json()
        if results:
            return float(results[0]["lat"]), float(results[0]["lon"])
    except Exception as exc:
        print(f"  [warn] Geocoding failed for '{query}': {exc}")

    return None


# Geocode every event from Stage 3
geocoded_events = []
for event in events:
    coords = geocode_location(event.get("city"), event.get("state"))
    if coords:
        event["latitude"], event["longitude"] = coords
        geocoded_events.append(event)
    time.sleep(1.1)  # Nominatim requires <= 1 request per second

print(f"Successfully geocoded: {len(geocoded_events)} of {len(events)} events")

## Stage 5 — Render the folium map

The final stage converts the geocoded events into an interactive map. Each event becomes a colored marker; clicking the marker reveals a popup with the company name, filing date, summary, and a link to the underlying SEC filing.

### Background: visualization design choices

A small number of decisions in the code below collectively determine how readable the final map is. Each is worth understanding rather than treating as boilerplate.

**Tile choice: CartoDB Positron.** Folium can use any of several base map providers. CartoDB Positron is a deliberately muted gray-and-white style designed specifically for data visualization — its low visual contrast lets the data markers stand out instead of competing with detailed geography. The default OpenStreetMap tiles, by contrast, include heavy color and labeling that make small markers harder to see.

**Color encoding by event type.** The `EVENT_COLORS` dictionary assigns a distinct color to each event type, providing a visual taxonomy that the eye can decode at a glance: green for openings (positive), red for closings (negative), orange for relocations (neutral movement), blue for expansions (positive growth), gray for ambiguous "other" events. This mapping follows widely understood color conventions; choosing colors that match audience intuition reduces the cognitive load of reading the map.

**Popups rather than always-visible labels.** With twenty to thirty markers across the United States, always-visible labels would overlap and clutter the map. Popups (revealed only on click) keep the default view clean while making the underlying detail available on demand. This is a common pattern: the high-level view shows distribution and density; interaction reveals individual records.

**Centering on the data.** Setting the initial map view to the geographic mean of the events ensures most markers are visible without manual panning. For a US-focused dataset this typically lands near the center of the continental United States.

### Exercise S5 — Configure the folium marker loop

The cell below provides the fixed scaffolding: the `EVENT_COLORS` dictionary, the centering calculation, and the empty `folium.Map(...)` object. Your task is to write the loop that adds one marker per event.

**Required behavior for each marker:**

1. Look up the marker color from `EVENT_COLORS` using `event.get("event_type")`. Default to `"gray"` for unknown event types.
2. Compose a popup HTML string that displays, in order: the company name (bold), the filing date and event type (italic, on one line), the summary, and a link to the SEC filing (`event['url']`) that opens in a new tab. Wrap the whole popup in a fixed-width `<div>` of about 260 pixels.
3. Construct a `folium.Marker(...)` at `[event['latitude'], event['longitude']]` with the popup, a tooltip showing the company name, and `folium.Icon(color=color, icon="info-sign")`.
4. Call `.add_to(m)` so the marker attaches to the map.

Build the loop where indicated. The final `m` on its own line at the bottom of the cell renders the interactive map inline.

In [ ]:
import folium

EVENT_COLORS = {
    "opening": "green",
    "closing": "red",
    "relocation": "orange",
    "expansion": "blue",
    "other": "gray",
}

# Center the map on the geographic mean of the events
if geocoded_events:
    center_lat = sum(e["latitude"] for e in geocoded_events) / len(geocoded_events)
    center_lon = sum(e["longitude"] for e in geocoded_events) / len(geocoded_events)
else:
    center_lat, center_lon = 39.5, -98.35  # geographic center of contiguous US

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=4,
    tiles="CartoDB positron",
)

# TODO 1: Loop over geocoded_events.
# TODO 2: For each event, look up the color from EVENT_COLORS (default "gray").
# TODO 3: Build the popup HTML string per the four requirements above.
# TODO 4: Construct a folium.Marker(...) and call .add_to(m).

raise NotImplementedError("Complete Exercise S5 to render the map.")

m

# ══ SOLUTION ══

## Evaluation notes

After your first end-to-end run, record the following observations. They serve as sanity checks and as the starting point for the assignment.

1. **Total filings returned by the keyword search** — captured in `total_found`. This is the upper bound on candidate volume for the chosen window.
2. **Fraction classified as genuine location events** — the ratio `events / fetched_filings`. The recall/precision discussion in Stage 3 explains why this percentage is typically low and why that is acceptable. If the rate is below approximately 8 percent, the keyword set is too broad and `"new location"` is the highest-leverage phrase to remove first.
3. **Geocoding success rate** — `geocoded_events / events`. Low rates typically indicate that the LLM is returning city names that Nominatim cannot resolve (often international locations, neighborhood-level descriptors, or generic regional language).
4. **Approximate cost per run** — visible in the Anthropic API console under Usage. With Haiku 4.5 at one dollar per million input tokens, a typical 100-filing run costs approximately thirty cents.

### Possible refinements for future iterations

- Pre-curate a list of ten to fifteen well-known retailers or logistics firms and restrict the EDGAR search by ticker. This produces a smaller, more reliable candidate set suitable for a single lab session.
- Replace the broad keyword query with item-code filtering (8-K Item 2.05 for exit activities, Item 2.01 for asset acquisitions). This sacrifices recall for precision.
- Add a second LLM call that produces a 100-word executive summary of the geographic pattern across all events, printed below the map.
- Cache results in a local CSV so you can iterate on the visualization without re-running the API pipeline.
- Add a date-range slider to the folium map so users can filter events by filing date interactively.